# 05 · Ingestão Bronze — physical_itens_venda_caixa
**Projeto:** Estágio Engenharia de Dados · **Squad:** 3 — Batch · **Autor:** Alexandro Silva

Primeira camada da arquitetura Medallion. Lê o CSV da RAW **sem tratamento de dados**,
adiciona as colunas de auditoria e grava em Delta, em modo `append`, particionado por `ano` / `mes`.

## Regras e KPIs gerados por este notebook
A Bronze não aplica regras de negócio nem gera KPIs: as regras técnicas ficam na Silver
(notebook 06) e os KPIs na Gold. O que este notebook gera:

| Nº | O que é gerado | Descrição |
|---|---|---|
| 1 | Controle de reprocessamento | Só ingere se a data de modificação do CSV na RAW mudou desde a última carga |
| 2 | `bronze_ingested_at` | Data e hora da ingestão |
| 3 | `bronze_source_file` | Caminho do arquivo de origem |
| 4 | `ano`, `mes` | Partição pela `dt_venda`, obtida de `physical_vendas_caixa` (a tabela de itens não tem data) |
| 5 | Registro no controle | Arquivo, data de modificação, horário e quantidade de linhas de cada carga |

## Origem e destino
| Item | Definição |
|---|---|
| Origem | `abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_itens_venda_caixa.csv` (somente leitura) |
| Destino | `abfss://squad3@internshipdatalake.dfs.core.windows.net/bronze/physical_itens_venda_caixa` |
| Controle | `abfss://squad3@internshipdatalake.dfs.core.windows.net/controle/bronze_processados` |
| Formato / modo | Delta / `append` |
| Frequência | semanal; a carga nova chega às segundas-feiras |

## Etapas
1. Credenciais e caminhos
2. Verificação de alteração do arquivo na RAW
3. Leitura da RAW e colunas de auditoria e partição
4. Gravação na Bronze e registro no controle
5. Validação

## 1. Credenciais e caminhos
Carrega as credenciais do Service Principal a partir do `.env` (nunca escritas no notebook) e define
os caminhos de origem, destino e controle. As opções OAuth são passadas em cada leitura e gravação,
porque o Databricks Serverless não permite configurá-las na sessão.

In [0]:
# Célula 1 — Credenciais e caminhos
from dotenv import load_dotenv
import os

load_dotenv("/Workspace/Users/alexandrofs31@gmail.com/.env")

CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
TENANT_ID       = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT", "internshipdatalake")

faltando = [n for n, v in {"ADLS_CLIENT_ID": CLIENT_ID, "ADLS_TENANT_ID": TENANT_ID,
                           "ADLS_CLIENT_SECRET": CLIENT_SECRET}.items() if not v]
if faltando:
    raise ValueError(f"Variáveis ausentes no .env: {faltando}")

TABELA        = "physical_itens_venda_caixa"
TABELA_VENDAS = "physical_vendas_caixa"

BASE_RAW   = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net"      # somente leitura
BASE_SQUAD = f"abfss://squad3@{STORAGE_ACCOUNT}.dfs.core.windows.net"

CAMINHO_RAW_ITENS  = f"{BASE_RAW}/batch-data/{TABELA}.csv"
CAMINHO_RAW_VENDAS = f"{BASE_RAW}/batch-data/{TABELA_VENDAS}.csv"
CAMINHO_BRONZE     = f"{BASE_SQUAD}/bronze/{TABELA}"
CAMINHO_CONTROLE   = f"{BASE_SQUAD}/controle/bronze_processados"   # controle compartilhado da squad

sufixo = f"{STORAGE_ACCOUNT}.dfs.core.windows.net"
opcoes_adls = {
    f"fs.azure.account.auth.type.{sufixo}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{sufixo}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{sufixo}": CLIENT_ID,
    f"fs.azure.account.oauth2.client.secret.{sufixo}": CLIENT_SECRET,
    f"fs.azure.account.oauth2.client.endpoint.{sufixo}": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

print(f"📂 Origem   : {CAMINHO_RAW_ITENS}")
print(f"📦 Destino  : {CAMINHO_BRONZE}")
print(f"🗂️  Controle : {CAMINHO_CONTROLE}")

## 2. Verificação de alteração do arquivo na RAW
Consulta apenas os metadados do CSV (caminho e data de modificação) e compara com o histórico do
controle. Se aquela versão do arquivo já foi ingerida, as etapas 3 e 4 não gravam nada. Na primeira
execução o controle ainda não existe e é criado na etapa 4.

In [0]:
# Célula 2 — O arquivo da RAW mudou desde a última carga?
from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime

# Versão atual do arquivo (só metadados; o conteúdo não é lido aqui)
meta = (spark.read.format("binaryFile").options(**opcoes_adls)
    .load(CAMINHO_RAW_ITENS)
    .select("path", "modificationTime")
    .first())

arquivo     = meta["path"]
modificacao = meta["modificationTime"]

# Histórico de cargas. Na primeira execução o controle ainda não existe.
try:
    df_controle = spark.read.format("delta").options(**opcoes_adls).load(CAMINHO_CONTROLE)
    ja_processado = (df_controle
        .filter((F.col("tabela_destino") == TABELA)
                & (F.col("bronze_source_file") == arquivo)
                & (F.col("source_modification_time") == F.lit(modificacao)))
        .limit(1).count() > 0)
except Exception:
    ja_processado = False
    print("ℹ️  Controle ainda não existe; será criado nesta carga.")

print(f"📄 Arquivo           : {arquivo}")
print(f"🕒 Última modificação : {modificacao}")
print("⏭️  Esta versão já foi ingerida: nada a fazer." if ja_processado
      else "🆕 Versão nova do arquivo: será ingerida.")

## 3. Leitura da RAW e colunas de auditoria e partição
Lê o CSV como veio, sem limpar nem corrigir valores. O único cruzamento é com `physical_vendas_caixa`,
usado apenas para obter `dt_venda` e montar a partição `ano` / `mes`; a `dt_venda` em si não é gravada.

In [0]:
# Célula 3 — Leitura da RAW, colunas de auditoria e de partição (sem tratamento de dados)

if not ja_processado:
    df_itens = (spark.read
        .options(**opcoes_adls)
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(CAMINHO_RAW_ITENS)
        .withColumn("bronze_source_file", F.col("_metadata.file_path")))

    # dt_venda vem da tabela de vendas e é usada apenas para montar a partição
    df_vendas = (spark.read
        .options(**opcoes_adls)
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(CAMINHO_RAW_VENDAS)
        .select("id_transacao", "dt_venda"))

    df_bronze = (df_itens
        .join(df_vendas, on="id_transacao", how="left")
        .withColumn("ano", F.year(F.col("dt_venda").cast("date")))
        .withColumn("mes", F.month(F.col("dt_venda").cast("date")))
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .drop("dt_venda"))

    qtd_registros = df_itens.count()
    print(f"✅ Linhas lidas da RAW: {qtd_registros:,}")
    print("Colunas da Bronze:")
    for c in df_bronze.columns:
        print(f"   • {c}")

## 4. Gravação na Bronze e registro no controle
Adiciona a carga à Bronze com `append` e, em seguida, registra a versão do arquivo no controle.
Como cada carga adiciona o arquivo inteiro, a Bronze guarda o histórico de todas as cargas; a
deduplicação por `id_item_venda` é feita na Silver.

In [0]:
# Célula 4 — Gravação na Bronze (Delta, append) e registro no controle

if not ja_processado:
    (df_bronze.write
        .format("delta")
        .options(**opcoes_adls)
        .mode("append")
        .partitionBy("ano", "mes")
        .save(CAMINHO_BRONZE))
    print(f"✅ Bronze: {qtd_registros:,} linhas adicionadas em {CAMINHO_BRONZE}")

    df_registro = spark.createDataFrame([Row(
        bronze_source_file=arquivo,
        source_modification_time=modificacao,
        tabela_destino=TABELA,
        bronze_processed_at=datetime.now(),
        qtd_registros=qtd_registros,
    )])
    (df_registro.write
        .format("delta")
        .options(**opcoes_adls)
        .mode("append")
        .save(CAMINHO_CONTROLE))
    print("✅ Carga registrada no controle.")
else:
    print("⏭️  Nenhuma gravação: o arquivo não mudou desde a última carga.")

## 5. Validação
Confere o que está gravado: total de linhas, linhas por carga, linhas por partição e o histórico do
controle para esta tabela. Cada carga semanal deve aparecer uma única vez em "Linhas por carga".

In [0]:
# Célula 5 — Validação da Bronze

df_val = spark.read.format("delta").options(**opcoes_adls).load(CAMINHO_BRONZE)

print(f"✅ Total de linhas na Bronze: {df_val.count():,}")

print("\n📥 Linhas por carga (cada carga semanal aparece em uma linha):")
(df_val.groupBy("bronze_ingested_at", "bronze_source_file").count()
    .orderBy("bronze_ingested_at").show(truncate=False))

print("📅 Linhas por partição:")
df_val.groupBy("ano", "mes").count().orderBy("ano", "mes").show(60, truncate=False)

print("🗂️  Histórico do controle para esta tabela:")
(spark.read.format("delta").options(**opcoes_adls).load(CAMINHO_CONTROLE)
    .filter(F.col("tabela_destino") == TABELA)
    .orderBy(F.col("bronze_processed_at").desc())
    .show(truncate=False))

df_val.printSchema()